In [1]:
# import subprocess
# result = subprocess.run(["which", "tesseract"], capture_output=True, text=True)
# print(result.stdout)  # shows the actual path

# 1. Basic PDF Preprocessing with Unstructured

This extracts structured semantic elements from PDF.

In [1]:
# import os
# os.environ["PATH"] += ":/opt/homebrew/bin/tesseract"  # Apple Silicon M1/M2/M3

import os
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ["PATH"]

In [2]:
from unstructured.partition.pdf import partition_pdf

/Users/adityabhagwat/Projects/Unstructured-io-Document-Processing-Pipeline/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
elements = partition_pdf(
    filename="Training_Data/AML_NOTES_UNIT_1_2_3_4_5_merged.pdf",
    # Best strategy for RAG pipelines
    strategy="hi_res",
    # OCR languages
    languages=["eng"],
    # Keep metadata
    include_metadata=True,
    # Detect tables
    infer_table_structure=True,
    # Extract images/tables if needed
    extract_images_in_pdf=False,
)

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 18245.18it/s]


In [6]:
print(f"Total elements: {len(elements)}")

Total elements: 853


In [7]:
for el in elements[:50]:
    print(type(el))
    print(el.text)
    print(el.metadata)
    print("=" * 80)

<class 'unstructured.documents.elements.Title'>
Unsupervised Learning :
<class 'unstructured.documents.elements.Title'>
What is Unsupervised Learning?
<class 'unstructured.documents.elements.NarrativeText'>
As the name suggests, unsupervised learning is a machine learning technique in which models are not supervised using training dataset. Instead, models itself find the hidden patterns and insights from the given data. It can be compared to learning which takes place in the human brain while learning new things. It can be defined as:
<class 'unstructured.documents.elements.NarrativeText'>
Unsupervised learning is a type of machine learning in which models are trained using unlabelled dataset and are allowed to act on that data without any supervision.
<class 'unstructured.documents.elements.NarrativeText'>
Unsupervised learning cannot be directly applied to a regression or classification problem because unlike supervised learning, we have the input data but no corresponding output dat

# 2. Normalize the Output

Unstructured returns heterogeneous element objects:

* Title
* NarrativeText
* ListItem
* Table
* Header
* Footer
* etc.

You should normalize them into ONE stable schema.

In [9]:
normalized_docs = []

for idx, el in enumerate(elements):

    doc = {
        "id": f"doc_{idx}",
        "type": el.category,
        "text": el.text,
        "page_number": getattr(el.metadata, "page_number", None),
        "filename": getattr(el.metadata, "filename", None),
        "languages": getattr(el.metadata, "languages", None),
        "coordinates": str(getattr(el.metadata, "coordinates", None)),
        "source": "sample.pdf",
    }
    normalized_docs.append(doc)

In [10]:
print(normalized_docs[0])

{'id': 'doc_0', 'type': 'Title', 'text': 'Unsupervised Learning :', 'page_number': 1, 'filename': 'AML_NOTES_UNIT_1_2_3_4_5_merged.pdf', 'languages': ['eng'], 'coordinates': 'CoordinatesMetadata(points=((np.float64(455.00001093749995), np.float64(350.380126953125)), (np.float64(455.00001093749995), np.float64(433.4489440917969)), (np.float64(1287.67214583046), np.float64(433.4489440917969)), (np.float64(1287.67214583046), np.float64(350.380126953125))), system=<unstructured.documents.coordinates.PixelSpace object at 0x30eb65d30>)', 'source': 'sample.pdf'}


# 3. Chunk the Document Properly

Use Unstructured chunking because it preserves semantic structure.

Why chunk_by_title() is better:

* respects headings
* preserves sections
* prevents chunk mixing across topics
* ideal for PDFs

In [11]:
from unstructured.chunking.title import chunk_by_title

chunks = chunk_by_title(
    elements,
    max_characters=1200,
    new_after_n_chars=1000,
    # combine tiny sections
    combine_text_under_n_chars=200,
)

In [12]:
print(f"Total chunks: {len(chunks)}")

Total chunks: 140


In [13]:
for chunk in chunks[:3]:
    print(chunk.text)
    print("=" * 80)

Unsupervised Learning :

What is Unsupervised Learning?

As the name suggests, unsupervised learning is a machine learning technique in which models are not supervised using training dataset. Instead, models itself find the hidden patterns and insights from the given data. It can be compared to learning which takes place in the human brain while learning new things. It can be defined as:

Unsupervised learning is a type of machine learning in which models are trained using unlabelled dataset and are allowed to act on that data without any supervision.

Unsupervised learning cannot be directly applied to a regression or classification problem because unlike supervised learning, we have the input data but no corresponding output data. The goal of unsupervised learning is to find the underlying structure of dataset, group that data according to similarities, and represent that dataset in a compressed format.
Example: Suppose the unsupervised learning algorithm is given an input dataset co

# 4. Create Embeddings

Using SentenceTransformer model name "sentence-transformers/all-MiniLM-L6-v2" to create embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_texts = [chunk.text for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)